# 01 — Document Ingestion, Chunking, Embeddings & Indexing

Real knowledge base: **5 government schemes** (PM-KISAN, PMFBY, KCC, Agriculture
Infrastructure Fund, AP Annadata Sukhibhava), sourced via live web search across two sessions,
Real knowledge base: **3 government schemes** (PM-KISAN, PMFBY, KCC), sourced via live web search
in Batch 6, paraphrased into `data/raw/government_schemes/*.json` with `source_url` + `verified_at`
on every document (`docs/rag/source-policy.md`). Agriculture Infrastructure Fund and AP state schemes
are explicit `_STUB.json` files — excluded from the index on purpose, not silently missing.

**Sandbox note:** embeddings fall back to TF-IDF (`ai/rag/src/embeddings.py`) because `huggingface.co`
is outside this sandbox's network whitelist — same category of limitation as the disease-detection
pretrained weights. `used_semantic_embeddings` is returned by every call so this is never silently
confused with the real ADR-004 multilingual model, which works normally on unrestricted internet.

In [1]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai" / "common" / "config.py").exists())
sys.path.insert(0, str(_root))
from ai.rag.src.loaders import load_scheme_documents
from ai.rag.src.chunking import chunk_all
from ai.rag.src.retriever import build_index

docs = load_scheme_documents(_root / "data/raw/government_schemes")
print(f"Loaded {len(docs)} real, sourced scheme documents:")
for d in docs:
    print(f"  - {d['scheme_name']}  (source: {d['source_url']}, verified {d['verified_at']})")

07:01:28 | INFO     | faiss.loader | Loading faiss with AVX512 support.


07:01:28 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")


07:01:28 | INFO     | faiss.loader | Loading faiss with AVX2 support.


07:01:28 | INFO     | faiss.loader | Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")


07:01:28 | INFO     | faiss.loader | Loading faiss.


07:01:28 | INFO     | faiss.loader | Successfully loaded faiss.


Loaded 5 real, sourced scheme documents:
  - Agriculture Infrastructure Fund (AIF)  (source: https://agriinfra.dac.gov.in, verified 2026-08-16)
  - Annadata Sukhibhava (Andhra Pradesh) - successor to YSR Rythu Bharosa  (source: https://ysrrythubharosa.ap.gov.in, verified 2026-08-16)
  - KCC (Kisan Credit Card)  (source: https://fasalrin.gov.in, verified 2026-08-13)
  - PM-KISAN (Pradhan Mantri Kisan Samman Nidhi)  (source: https://pmkisan.gov.in, verified 2026-08-13)
  - PMFBY (Pradhan Mantri Fasal Bima Yojana)  (source: https://pmfby.gov.in, verified 2026-08-13)


## 1. Chunk

In [2]:
chunks = chunk_all(docs)
print(f"{len(chunks)} chunks (3 per scheme: summary / eligibility / confidence_caveat)")
chunks[0]

15 chunks (3 per scheme: summary / eligibility / confidence_caveat)


{'scheme_id': 'agriculture-infrastructure-fund',
 'scheme_name': 'Agriculture Infrastructure Fund (AIF)',
 'jurisdiction': 'Central',
 'source_url': 'https://agriinfra.dac.gov.in',
 'verified_at': '2026-08-16',
 'section': 'summary',
 'text': "AIF is a central financing facility - not a direct cash-transfer scheme - for building post-harvest and community-farming infrastructure: warehouses, cold chains, primary processing units, custom hiring centres for farm machinery, and similar projects. Launched in August 2020 under Aatmanirbhar Bharat with a Rs 1 lakh crore corpus, running through FY2032. Loans up to Rs 2 crore per project get a 3% per annum interest subvention for up to 7 years (capped so the effective rate doesn't exceed about 9%), plus credit-guarantee coverage under CGTMSE so no collateral is required up to that limit - the government pays the guarantee fee. There is no direct cash subsidy; the benefit is entirely through cheaper borrowing."}

## 2. Embed + index

In [3]:
store, used_semantic = build_index(chunks)
print(f"Index built with {store.index.ntotal} vectors, dim={store.index.d}")
print(f"used_semantic_embeddings = {used_semantic}  (False = TF-IDF sandbox fallback)")

07:01:36 | INFO     | sentence_transformers.base.model | No device provided, using cpu


07:01:36 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 403 Forbidden"


07:01:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 403 Forbidden"


07:01:36 | INFO     | sentence_transformers.base.model | No modules.json found for sentence-transformers/paraphrase-multilingual-mpnet-base-v2, initializing a new SentenceTransformer model.


07:01:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/adapter_config.json "HTTP/1.1 403 Forbidden"


07:01:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config.json "HTTP/1.1 403 Forbidden"


07:01:36 | WARNING  | ai.rag.src.embeddings | Could not load sentence-transformers/paraphrase-multilingual-mpnet-base-v2 (OSError) - huggingface.co is outside this sandbox's whitelisted network. Falling back to TF-IDF for THIS SANDBOX RUN ONLY - not multilingual, not ADR-004. Label results accordingly.


Index built with 15 vectors, dim=1087
used_semantic_embeddings = False  (False = TF-IDF sandbox fallback)


## 3. Save index

In [4]:
store.save(_root / "ai/rag/models/scheme_index_v1")
print("Saved to ai/rag/models/scheme_index_v1/")

Saved to ai/rag/models/scheme_index_v1/


## Next: `02_retrieval_evaluation.ipynb`

**Status: IMPLEMENTED.** Real 3-scheme, 9-chunk index built and saved above.